[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ItsNotAILABS/PARALLAX-Exchange-Clearinghouse/blob/main/examples/02_query_clearinghouse.ipynb)

# 02 — Query the Phantom Clearinghouse

This notebook demonstrates the **Phantom Clearinghouse** settlement system:
- Execute trades on the exchange
- Settle all fills in one heartbeat (873ms)
- View settlement proofs (cryptographic finality)
- Inspect net positions after multilateral netting
- Verify zero gas fees on every settlement

In [ ]:
!pip install -q numpy pandas matplotlib

import time
import hashlib
import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict
from enum import Enum

# PHI constants
PHI = 1.6180339887498948482
PHI_INV = 1.0 / PHI
PHI_INV_3 = PHI_INV ** 3
HEARTBEAT_MS = (PHI ** 4) * (1000.0 / 7.83)

print(f"Heartbeat interval: {HEARTBEAT_MS:.1f}ms")
print("✅ Ready")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Exchange + Clearinghouse Classes (compact version)
# ═══════════════════════════════════════════════════════════════

class OrderSide(Enum):
    BUY = "buy"
    SELL = "sell"

@dataclass
class Fill:
    fill_id: int
    pair_id: str
    buyer: str
    seller: str
    price: float
    quantity: float
    timestamp: float
    gas_fee: float = 0.0

@dataclass
class SettlementRecord:
    settlement_id: int
    fill_id: int
    pair_id: str
    buyer: str
    seller: str
    base_amount: float
    quote_amount: float
    settlement_beat: int
    proof: str
    gas_fee: float = 0.0
    status: str = "settled"

class PhantomClearinghouse:
    """Real-Time Multi-Asset Clearing & Settlement Engine"""
    
    def __init__(self):
        self.settlement_counter = 0
        self.settlements: List[SettlementRecord] = []
        self.positions: Dict[str, Dict[str, float]] = {}
        self.beat_counter = 0
        self.netting_history: List[dict] = []
    
    def settle_fills(self, fills: List[Fill]) -> List[SettlementRecord]:
        """Settle all fills in one heartbeat — 873ms cryptographic finality"""
        self.beat_counter += 1
        new_settlements = []
        
        for fill in fills:
            self.settlement_counter += 1
            proof_data = f"{fill.fill_id}:{fill.buyer}:{fill.seller}:{fill.price}:{fill.quantity}:{self.beat_counter}"
            proof = hashlib.sha256(proof_data.encode()).hexdigest()[:16]
            
            record = SettlementRecord(
                settlement_id=self.settlement_counter,
                fill_id=fill.fill_id,
                pair_id=fill.pair_id,
                buyer=fill.buyer,
                seller=fill.seller,
                base_amount=fill.quantity,
                quote_amount=fill.quantity * fill.price,
                settlement_beat=self.beat_counter,
                proof=proof
            )
            self.settlements.append(record)
            new_settlements.append(record)
            
            # Update positions
            pair = fill.pair_id.split("_")
            base, quote = pair[0], pair[1]
            for principal in [fill.buyer, fill.seller]:
                if principal not in self.positions:
                    self.positions[principal] = {}
            
            self.positions[fill.buyer][base] = self.positions[fill.buyer].get(base, 0) + fill.quantity
            self.positions[fill.buyer][quote] = self.positions[fill.buyer].get(quote, 0) - fill.quantity * fill.price
            self.positions[fill.seller][base] = self.positions[fill.seller].get(base, 0) - fill.quantity
            self.positions[fill.seller][quote] = self.positions[fill.seller].get(quote, 0) + fill.quantity * fill.price
        
        # Record netting
        netting = self.get_netting_summary()
        self.netting_history.append({"beat": self.beat_counter, **netting})
        return new_settlements
    
    def get_netting_summary(self) -> dict:
        gross = sum(abs(pos) for positions in self.positions.values() for pos in positions.values())
        net = sum(abs(sum(positions.values())) for positions in self.positions.values())
        reduction = 1.0 - (net / gross) if gross > 0 else 0.0
        return {"gross_obligations": gross, "net_obligations": net, "reduction_ratio": reduction}
    
    def get_position_dataframe(self) -> pd.DataFrame:
        rows = []
        for principal, tokens in self.positions.items():
            for token, amount in tokens.items():
                rows.append({"principal": principal, "token": token, "net_position": amount})
        return pd.DataFrame(rows)

print("✅ Clearinghouse defined")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 1: Generate simulated fills (as if exchange matched them)
# ═══════════════════════════════════════════════════════════════

# Simulate a trading session with multiple participants
fills = [
    Fill(1, "ICP_USDT", "alice", "bob", 12.45, 50.0, time.time()),
    Fill(2, "ICP_USDT", "carol", "alice", 12.50, 30.0, time.time()),
    Fill(3, "ICP_USDT", "bob", "dave", 12.48, 25.0, time.time()),
    Fill(4, "ICP_USDT", "dave", "carol", 12.52, 40.0, time.time()),
    Fill(5, "ICP_USDT", "alice", "dave", 12.47, 60.0, time.time()),
]

print(f"Generated {len(fills)} fills for settlement")
print(f"Participants: alice, bob, carol, dave")
print(f"\nFill Summary:")
for f in fills:
    print(f"  #{f.fill_id}: {f.buyer} buys {f.quantity:.0f} ICP @ ${f.price:.2f} from {f.seller}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 2: Settle all fills in one heartbeat
# ═══════════════════════════════════════════════════════════════

clearinghouse = PhantomClearinghouse()
settlements = clearinghouse.settle_fills(fills)

print("=" * 60)
print(f"⚡ SETTLEMENT COMPLETE — Beat #{clearinghouse.beat_counter}")
print(f"   Finality: {HEARTBEAT_MS:.0f}ms (φ⁴ × 1000/7.83Hz)")
print("=" * 60)

for s in settlements:
    print(f"\n  Settlement #{s.settlement_id}:")
    print(f"    Fill: #{s.fill_id} | {s.buyer} ← {s.base_amount:.0f} ICP ← {s.seller}")
    print(f"    Quote: ${s.quote_amount:.2f} USDT transferred")
    print(f"    Proof: 0x{s.proof}")
    print(f"    Gas:   ${s.gas_fee:.2f} ✨ ZERO")
    print(f"    Status: {s.status}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 3: Query net positions
# ═══════════════════════════════════════════════════════════════

print("NET POSITIONS (after multilateral netting):")
print("=" * 60)

df = clearinghouse.get_position_dataframe()
pivot = df.pivot_table(index='principal', columns='token', values='net_position', fill_value=0)
print(pivot.to_string())

print(f"\n\nNETTING EFFICIENCY:")
netting = clearinghouse.get_netting_summary()
print(f"  Gross Obligations: ${netting['gross_obligations']:,.2f}")
print(f"  Net Obligations:   ${netting['net_obligations']:,.2f}")
print(f"  Reduction:         {netting['reduction_ratio']:.1%}")
print(f"\n  → Netting reduced settlement obligations by {netting['reduction_ratio']:.1%}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 4: Verify settlement proofs
# ═══════════════════════════════════════════════════════════════

print("SETTLEMENT PROOF VERIFICATION:")
print("=" * 60)

all_valid = True
for s in clearinghouse.settlements:
    # Recompute proof
    proof_data = f"{s.fill_id}:{s.buyer}:{s.seller}:{fills[s.fill_id-1].price}:{s.base_amount}:{s.settlement_beat}"
    expected_proof = hashlib.sha256(proof_data.encode()).hexdigest()[:16]
    valid = expected_proof == s.proof
    all_valid = all_valid and valid
    status = "✅ VALID" if valid else "❌ INVALID"
    print(f"  Settlement #{s.settlement_id}: {status}  proof=0x{s.proof}")

print(f"\n{'✅ All settlement proofs verified — cryptographic finality confirmed' if all_valid else '❌ Proof verification failed'}")
print(f"\nTotal gas fees paid across all settlements: $0.00")
print(f"Settlement time: {HEARTBEAT_MS:.0f}ms per beat")